In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()

xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()
dids = df['did'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20250501T002002_V202602220258_0545010501.fits.gz']

In [10]:
file = files[4]

with fits.open(file) as hdul:
    header = hdul[0].header
    data = hdul[0].data

did = int(file.split('_')[-1].split('.')[0])
data = undistort(data, header, xd, yd)

In [11]:
view = View.from_header(header)
view.update(crota=view.crota + 0.29, xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0] + 0.28)

In [12]:
V = view.velocity(mu_thr=0.0) / 1000
V -= np.nanmean(V)

plt.figure(figsize=(10,10))
plt.imshow(V, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [13]:
V_ = data.copy()
V_[np.isnan(V)] = np.nan
V_ -= np.nanmean(V_)

plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [14]:
t = ~np.isnan(V)
x, y = V[t], V_[t]
t = np.where(np.abs(y - x) < 0.5)
x, y = x[t], y[t]

k = np.nanmean(x * y) / np.nanmean(x ** 2)
print(k)

plt.figure(figsize=(10,10))
plt.plot([-2,2], [-2,2], color='black', lw=0.5)
plt.scatter(V, V_, s=0.001)
plt.plot([-2,2], [-2 * k, 2 * k], '--', color='black', lw=1)

plt.xlim(-2,2)
plt.ylim(-2,2)

plt.tight_layout()

0.9423939174109446


In [16]:
plt.figure(figsize=(10,10))
plt.imshow(V_ - V * k, 'seismic', vmin=-1, vmax=1)

plt.tight_layout()

In [39]:
(np.arctan(0.015) - np.arcsin(0.015)) * 180 / np.pi * 3600

np.float64(-0.3480522881073428)

In [13]:
header['OBS_VR'] * (1 - np.cos(0.005))

np.float64(0.15630260858420247)

In [50]:
(np.tan(0.01) - 0.01) * 180 / np.pi * 3600

np.float64(0.06875768572457211)